# `groundinsight` — Inverse rho-f model determination

This notebook walks through the new `groundinsight.analysis.inverse_rho_f`
module step by step. The bus grounding impedance is parameterised in the
canonical linear rho-f form

$$
Z(\rho, f) = k_1\,\rho + (k_2 + j k_3)\,f + (k_4 + j k_5)\,\rho\,f
$$

with five real parameters `k = (k1, k2, k3, k4, k5)` -- the same form that
`groundmeas.services.analytics.rho_f_model` fits to measured impedance
points.

The goal is the *inverse* problem: given a fully built network, a list of
selected buses and an EPR limit `u_limit`, find the rho-f characteristic
that the network can tolerate while keeping the RMS earth potential rise
below `u_limit` at every bus when each bus is swept as the active fault.

Three public entry points are demonstrated below:

1. **`evaluate_max_epr_under_k`** — direct forward evaluation of one `k`.
2. **`find_max_rho_f_scaling`** — log-bisection of a uniform scaling
   factor along a reference vector `k_ref`.
3. **`select_rho_f_from_catalog`** — pick the feasible candidates from a
   user-supplied catalog of rho-f models.


## 1. Setup

Build a small three-bus medium-voltage line as a running example:
`b0` (source) -- cable -- `b1` -- cable -- `b2`. The cable has a fully
coupled shield (`mutual_impedance_formula` close to the self
formula) so the reduction effect is visible. The bus impedance shape
will be overwritten on the fly by the inversion routines, so the
`BusType.impedance_formula` here just provides a placeholder.


In [9]:
import polars as pl

import groundinsight as gi
from groundinsight.models.core_models import BusType, BranchType
from groundinsight.analysis import (
    evaluate_max_epr_under_k,
    find_max_rho_f_scaling,
    select_rho_f_from_catalog,
)


def build_demo_net() -> gi.Network:
    bt = BusType(
        name="DemoBus",
        description="Placeholder; bus impedance is overwritten by the inversion.",
        system_type="Grounded",
        voltage_level=20.0,
        impedance_formula="rho * 0.1 + 0*f",
    )
    brt = BranchType(
        name="DemoCable",
        description="20 kV XLPE cable, full mutual coupling.",
        grounding_conductor=True,
        self_impedance_formula="(0.25 + I*0.6)*l",
        mutual_impedance_formula="(0.0 + I*0.6)*l",
    )
    net = gi.create_network(name="DemoNet", frequencies=[50, 250])
    for name in ("b0", "b1", "b2"):
        gi.create_bus(name=name, type=bt, network=net,
                      specific_earth_resistance=100.0)
    gi.create_branch(name="b01", type=brt, from_bus="b0", to_bus="b1",
                     length=1.0, network=net)
    gi.create_branch(name="b12", type=brt, from_bus="b1", to_bus="b2",
                     length=1.0, network=net)
    gi.create_source(name="src", bus="b0",
                     values={50: 2000.0, 250: 60.0}, network=net)
    return net


net = build_demo_net()
print("Network:", net.name, "| buses:", list(net.buses), "| freqs:", net.frequencies)


Network: DemoNet | buses: ['b0', 'b1', 'b2'] | freqs: [50.0, 250.0]


## 2. `evaluate_max_epr_under_k` -- forward evaluation

Pick a single `k` and observe the resulting RMS EPR at each swept bus.
Each bus is set as the active fault one by one; the reported EPR is the
RMS over all simulation frequencies at that bus.

The configuration `Source==Fault` (here `b0`) is electrically degenerate
-- the injected current cancels at the same node, so the EPR is zero.
We include `b0` in the sweep on purpose to make the behaviour visible.


In [10]:
# k_ref reproducing the placeholder Z = 0.01 * rho.
k = (0.1, 0.0, 0.0, 0.0, 0.0)
eprs = evaluate_max_epr_under_k(net, ["b0", "b1", "b2"], k=k)
for bus, epr in eprs.items():
    print(f"  EPR_RMS @ {bus}: {epr:.3f} V")
print("Max swept EPR:", max(eprs.values()), "V")


  EPR_RMS @ b0: 0.000 V
  EPR_RMS @ b1: 165.331 V
  EPR_RMS @ b2: 487.190 V
Max swept EPR: 487.19036802281556 V


In [11]:
# Add a frequency-coupled imaginary term and observe how the EPR grows.
k_with_f = (0.1, 0.0, 1e-3, 0.0, 1e-5)
eprs_f = evaluate_max_epr_under_k(net, ["b0", "b1", "b2"], k=k_with_f)
for bus, epr in eprs_f.items():
    print(f"  EPR_RMS @ {bus}: {epr:.3f} V")


  EPR_RMS @ b0: 0.000 V
  EPR_RMS @ b1: 165.299 V
  EPR_RMS @ b2: 486.914 V


## 3. `find_max_rho_f_scaling` -- 1-D head-room search

Given a reference rho-f vector `k_ref` (e.g. a fit produced by
`groundmeas.rho_f_model` from measured impedance points), how much can we
scale it before the EPR limit is violated? `find_max_rho_f_scaling`
log-bisects the largest factor `c` such that `k = c * k_ref` keeps
`max_i u_EPR_RMS(bus_i) <= u_limit`.


In [13]:
k_ref = (0.1, 0.0, 0.0, 0.0, 0.0)
u_limit = 550.0  # volts

result = find_max_rho_f_scaling(
    net, ["b0", "b1", "b2"],
    u_limit=u_limit,
    k_ref=k_ref,
    tol_rel=1e-4,
)
print(f"c_max               = {result['c_max']:.5f}")
print(f"k_max               = {result['k_max']}")
print(f"max EPR_RMS @ c_max = {result['max_epr_rms_at_c_max']:.5f} V")
print(f"per-bus EPR        : {result['epr_rms_per_bus_at_c_max']}")
print(f"iterations         : {result['iterations']}")


c_max               = 1000.00000
k_max               = (100.0, 0.0, 0.0, 0.0, 0.0)
max EPR_RMS @ c_max = 500.21244 V
per-bus EPR        : {'b0': 0.0, 'b1': 166.7402602638735, 'b2': 500.2124431866835}
iterations         : 0


## 4. `select_rho_f_from_catalog` -- pick from a curated list

In practice you often have a *catalog* of plausible rho-f
characteristics -- e.g. one fit per soil class from `groundmeas`,
or generic textbook curves. `select_rho_f_from_catalog` evaluates every
candidate and reports a Polars DataFrame with the per-candidate maximum
EPR, the per-bus EPRs and a boolean `admissible` column flagging the
candidates that satisfy `u_limit`.

The default sort puts admissible candidates first, ordered by ascending
maximum EPR within each block (so the tightest-margin admissible
candidate is at the top, immediately above the closest miss).


In [15]:
catalog = {
    "very_dry":   (0.020, 0.0, 0.0, 0.0, 0.0),
    "dry_sand":   (0.010, 0.0, 1e-4, 0.0, 0.0),
    "loam":       (0.005, 0.0, 5e-4, 0.0, 1e-6),
    "wet_clay":   (0.002, 0.0, 1e-3, 0.0, 5e-6),
    "permafrost": (0.050, 0.0, 0.0, 0.0, 0.0),
}

df = select_rho_f_from_catalog(
    net,
    bus_names=["b0", "b1", "b2"],
    u_limit=300.0,
    candidates=catalog,
)
df


name,k1,k2,k3,k4,k5,max_epr_rms_V,admissible,epr_b0_V,epr_b1_V,epr_b2_V
str,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64
"""wet_clay""",0.002,0.0,0.001,0.0,0.000005,131.782692,true,0.0,90.222866,131.782692
"""loam""",0.005,0.0,0.0005,0.0,0.000001,255.800516,true,0.0,133.19406,255.800516
"""dry_sand""",0.01,0.0,0.0001,0.0,0.0,360.210027,false,0.0,151.232967,360.210027
"""very_dry""",0.02,0.0,0.0,0.0,0.0,429.630954,false,0.0,159.339432,429.630954
"""permafrost""",0.05,0.0,0.0,0.0,0.0,473.323648,false,0.0,163.881387,473.323648


In [17]:
# Just the admissible names:
admissible = df.filter(pl.col("admissible"))["name"].to_list()
print("Admissible candidates:", admissible)


Admissible candidates: ['wet_clay', 'loam']


## 5. Variation: tightening `u_limit`

Halving `u_limit` should shrink the admissible set; the order of the
remaining candidates by EPR margin stays consistent.


In [18]:
df_tight = select_rho_f_from_catalog(
    net,
    bus_names=["b0", "b1", "b2"],
    u_limit=7.5,
    candidates=catalog,
)
df_tight


name,k1,k2,k3,k4,k5,max_epr_rms_V,admissible,epr_b0_V,epr_b1_V,epr_b2_V
str,f64,f64,f64,f64,f64,f64,bool,f64,f64,f64
"""wet_clay""",0.002,0.0,0.001,0.0,0.000005,131.782692,false,0.0,90.222866,131.782692
"""loam""",0.005,0.0,0.0005,0.0,0.000001,255.800516,false,0.0,133.19406,255.800516
"""dry_sand""",0.01,0.0,0.0001,0.0,0.0,360.210027,false,0.0,151.232967,360.210027
"""very_dry""",0.02,0.0,0.0,0.0,0.0,429.630954,false,0.0,159.339432,429.630954
"""permafrost""",0.05,0.0,0.0,0.0,0.0,473.323648,false,0.0,163.881387,473.323648


## 6. Notes and roadmap

- The 1-D scaling solver and the catalog selector both rely on the same
  `evaluate_max_epr_under_k` helper, so anything you can express as a
  catalog entry can also be probed directly with `evaluate_max_epr_under_k`
  for free.
- A full Pareto front in :math:`\mathbb{R}^5` (the natural extension of
  this problem) is on the roadmap; it will land as
  `find_max_rho_f_pareto_front` on top of the same helper.
- The configuration `Source==Fault` is degenerate (zero EPR by
  construction); when sweeping all buses, treat the source bus' EPR as
  uninformative.
